# Kaggle: Silero TTS склейка аудиодорожки из LLM JSONL

Что загрузить в Kaggle Dataset:

1. `gigachat_outputs.jsonl` — ответы LLM.
2. `3930158_events_std_clean.json` или `3930158_events_std.json` — события матча для определения тайма.

Ноутбук сохранит результат в `/kaggle/working/outputs/audio_match_...`:

- отдельные WAV реплик;
- WAV по таймам;
- полный WAV матча с тихим шумом в паузах;
- CSV-таблицы индекса реплик и размещения.


In [ ]:
# Если в Kaggle не хватает зависимостей, раскомментируй и выполни эту ячейку.
# Обычно torch/torchaudio уже есть, а omegaconf может отсутствовать.
!pip -q install omegaconf soundfile


In [27]:
from pathlib import Path
import json
import math
import re
import html

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
import torchaudio
from IPython.display import Audio, display


## 1. Настройки

Если Kaggle сам не найдёт файлы, явно впиши пути в `GIGACHAT_JSONL` и `EVENTS_JSON`.


In [28]:
KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
BASE_DIR = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path('.')

MATCH_ID = '3930158'
STRATEGY = 'related_stop_cap'

# Если надо, можно руками заменить после auto-find:
GIGACHAT_JSONL = None
EVENTS_JSON = None

AUDIO_OUT_DIR = BASE_DIR / 'outputs' / f'audio_match_{MATCH_ID}_{STRATEGY}_silero_xfast'
CLIPS_DIR = AUDIO_OUT_DIR / 'clips_by_comment'
TRACKS_DIR = AUDIO_OUT_DIR / 'stitched_tracks'
TABLES_DIR = AUDIO_OUT_DIR / 'tables'
for d in [AUDIO_OUT_DIR, CLIPS_DIR, TRACKS_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Silero settings
language = 'ru'

# ВАЖНО: в torch.hub.load параметр speaker означает ID МОДЕЛИ Silero,
# а не конкретный голос диктора. Голос задаётся ниже в apply_tts.
# Для русской речи берём мультиспикерную русскую модель.
model_id = 'v4_ru'

# Русскоязычные голоса для v4_ru/v5_ru обычно: baya, xenia, kseniya, eugene.
# aidar не используем: он часто звучит не как нейтральный русский комментатор.
speaker = 'baya'

sample_rate = 48000
rate = 'x-fast'

# На Kaggle GPU у Silero иногда падает из-за несовместимости CUDA/kernel image.
# Поэтому по умолчанию CPU-режим надёжнее. Если GPU точно работает, поставь FORCE_CPU=False.
FORCE_CPU = True
DEVICE = torch.device('cuda' if (torch.cuda.is_available() and not FORCE_CPU) else 'cpu')

# Stitching settings
AVOID_OVERLAP = True
MIN_GAP_SEC = 0.15
TAIL_SEC = 3.0
HALF_GAP_SEC = 6.0
NOISE_RMS = 0.0020
SPEECH_GAIN = 0.95
RANDOM_SEED = 42

# True -> пересинтезировать клипы при смене голоса/скорости, даже если wav уже есть.
REGENERATE_CLIPS = True

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda device:', torch.cuda.get_device_name(0))
print('force cpu:', FORCE_CPU)
print('device:', DEVICE)
print('silero model_id:', model_id)
print('voice speaker:', speaker)
print('regenerate clips:', REGENERATE_CLIPS)
print('out:', AUDIO_OUT_DIR)




torch: 2.10.0+cu128
cuda available: True
cuda device: Tesla T4
device: cuda
out: /kaggle/working/outputs/audio_match_3930158_related_stop_cap_silero_xfast


In [29]:
def find_one(patterns, root):
    hits = []
    for pat in patterns:
        hits.extend(root.rglob(pat))
    hits = sorted(set(hits))
    return hits[0] if hits else None

if GIGACHAT_JSONL is None:
    GIGACHAT_JSONL = find_one(['gigachat_outputs.jsonl', '*gigachat_outputs*.jsonl'], KAGGLE_INPUT if KAGGLE_INPUT.exists() else Path('.'))

if EVENTS_JSON is None:
    EVENTS_JSON = find_one([f'{MATCH_ID}_events_std_clean.json', f'{MATCH_ID}_events_std.json', '*events_std_clean.json', '*events_std.json'], KAGGLE_INPUT if KAGGLE_INPUT.exists() else Path('.'))

print('GIGACHAT_JSONL:', GIGACHAT_JSONL)
print('EVENTS_JSON    :', EVENTS_JSON)

if GIGACHAT_JSONL is None:
    raise FileNotFoundError('Не найден gigachat_outputs.jsonl в /kaggle/input')
if EVENTS_JSON is None:
    raise FileNotFoundError('Не найден events_std_clean/events_std json в /kaggle/input')


GIGACHAT_JSONL: /kaggle/input/datasets/angelinamyasnikova/rtrrrr/gigachat_outputs.jsonl
EVENTS_JSON    : /kaggle/input/datasets/angelinamyasnikova/rtrrrr/3930158_events_std_clean.json


## 2. Чтение LLM-комментариев и событий


In [30]:
def read_jsonl(path: Path):
    rows = []
    with Path(path).open(encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def to_seconds(ts):
    if not isinstance(ts, str):
        return None
    m = re.match(r'^(\d+):(\d+):(\d+)(?:\.(\d+))?$', ts.strip())
    if not m:
        return None
    hh, mm, ss, ms = m.groups()
    return int(hh) * 3600 + int(mm) * 60 + int(ss) + (float(f'0.{ms}') if ms else 0.0)


def sec_to_stamp(sec):
    sec = max(0.0, float(sec))
    mm = int(sec // 60)
    ss = int(sec % 60)
    ms = int(round((sec - int(sec)) * 1000))
    if ms == 1000:
        ss += 1
        ms = 0
    return f'{mm:02d}-{ss:02d}-{ms:03d}'

rows_raw = read_jsonl(Path(GIGACHAT_JSONL))
events_std = json.loads(Path(EVENTS_JSON).read_text(encoding='utf-8'))
events_by_id = {e.get('id'): e for e in events_std if e.get('id')}

print('raw rows:', len(rows_raw))
print('events:', len(events_std))


raw rows: 679
events: 3373


In [31]:
def row_period(row):
    for eid in row.get('chain_event_ids') or []:
        ev = events_by_id.get(eid)
        if ev and ev.get('period') is not None:
            return int(ev.get('period'))
    return None


def parsed_comment(row):
    p = row.get('parsed') if isinstance(row.get('parsed'), dict) else {}
    return {
        't_start': p.get('t_start'),
        't_end': p.get('t_end'),
        'commentary': (p.get('commentary') or '').strip(),
    }

comment_rows = []
for i, r in enumerate(rows_raw):
    if str(r.get('match_id')) != str(MATCH_ID):
        continue
    if r.get('strategy') != STRATEGY:
        continue
    pc = parsed_comment(r)
    if not pc['commentary']:
        continue
    period = row_period(r)
    t_start_sec = to_seconds(pc['t_start'])
    t_end_sec = to_seconds(pc['t_end'])
    if period is None or t_start_sec is None:
        continue
    comment_rows.append({
        'row_idx': i,
        'payload_idx': r.get('payload_idx'),
        'strategy': r.get('strategy'),
        'match_id': r.get('match_id'),
        'period': period,
        't_start': pc['t_start'],
        't_end': pc['t_end'],
        't_start_sec': t_start_sec,
        't_end_sec': t_end_sec if t_end_sec is not None else t_start_sec,
        'commentary': pc['commentary'],
        'selection': r.get('selection'),
        'chain_event_ids': r.get('chain_event_ids') or [],
    })

comments_df = pd.DataFrame(comment_rows).sort_values(['period', 't_start_sec', 'payload_idx']).reset_index(drop=True)
comments_df['comment_idx'] = np.arange(1, len(comments_df) + 1)
print('comments selected:', len(comments_df))
display(comments_df[['comment_idx','period','t_start','t_end','commentary']].head(20))
comments_df.to_csv(TABLES_DIR / 'comments_selected_for_tts.csv', index=False)


comments selected: 679


,comment_idx,period,t_start,t_end,commentary
0,1,1,00:00:01.191,00:00:04.165,"Хавертц начинает игру с центра поля, отдавая п..."
1,2,1,00:00:07.397,00:00:07.871,Хавертц делает пас низом после начального удар...
2,3,1,00:00:08.713,00:00:09.564,"Хендери делает передачу, но её блокирует Мюсиала."
3,4,1,00:00:10.048,00:00:10.048,Гендри блокирует удар после начального удара.
4,5,1,00:00:10.987,00:00:10.987,МакГрегор с первых секунд демонстрирует мастер...
5,6,1,00:00:13.928,00:00:17.112,Та получает мяч после начального удара и делае...
6,7,1,00:00:18.447,00:00:22.695,"Рюдигер делает пас низом на Tah, затем Tah так..."
7,8,1,00:00:23.672,00:00:26.784,"Кроос получает мяч от начального удара, продви..."
8,9,1,00:00:28.020,00:00:32.000,"Mittelstädt делает передачу низом на Крооса, з..."
9,10,1,00:00:33.290,00:00:38.592,На открытии встречи Германия быстро организует...


In [32]:
def period_duration_from_events(events, period):
    vals = []
    for e in events:
        if int(e.get('period') or -1) == int(period):
            s = to_seconds(e.get('timestamp'))
            if s is not None:
                vals.append(s)
    return max(vals) if vals else 0.0

period_durations = {p: period_duration_from_events(events_std, p) + TAIL_SEC for p in sorted(comments_df['period'].unique())}
period_durations


{np.int64(1): 2916.941, np.int64(2): 2922.8}

## 3. Загрузка Silero TTS


In [33]:
# --- Robust Silero TTS loading ---
# torch.hub.load у разных версий Silero возвращает либо model, либо tuple.
# Поэтому не распаковываем как `model, _ = ...`, а нормализуем вручную.

import warnings
warnings.filterwarnings('ignore', category=SyntaxWarning)

loaded = torch.hub.load(
    repo_or_dir='snakers4/silero-models',
    model='silero_tts',
    language=language,
    speaker=model_id,
    trust_repo=True,
)

model = loaded[0] if isinstance(loaded, tuple) else loaded

# У Silero wrapper может не быть eval(). Это нормально.
# .to(device) есть не у всех wrapper-объектов, поэтому проверяем.
if hasattr(model, 'to'):
    try:
        model.to(DEVICE)
    except Exception as e:
        print('model.to(device) failed, keep model as is:', repr(e))

print('loaded object:', type(model))
print('has apply_tts:', hasattr(model, 'apply_tts'))
assert hasattr(model, 'apply_tts'), 'Silero model object has no apply_tts; check torch.hub.load output'
print('loaded Silero model:', model_id, '| voice:', speaker, '| device:', DEVICE)



Using cache found in /root/.cache/torch/hub/snakers4_silero-models_master


loaded Silero on cuda


## 4. Генерация отдельных реплик


In [ ]:
def escape_ssml(text):
    return html.escape(str(text), quote=False)


def to_ssml(text, rate_value):
    return f"<speak><prosody rate='{rate_value}'>{escape_ssml(text)}</prosody></speak>"


def normalize_peak(wav, peak=0.98):
    max_abs = wav.abs().max().item() if wav.numel() else 0.0
    if max_abs > peak and max_abs > 0:
        wav = wav * (peak / max_abs)
    return wav


def _as_2d_float_tensor(wav):
    if not torch.is_tensor(wav):
        wav = torch.tensor(wav)
    if wav.dim() == 1:
        wav = wav.unsqueeze(0)
    return wav.to(torch.float32)


def synthesize_text(text):
    ssml = to_ssml(text, rate)

    # Для v4_ru/v5_ru передаём русский голос speaker='baya'/'kseniya'/...
    # Если конкретная версия модели не поддерживает ssml_text, fallback на plain text.
    try:
        wav = model.apply_tts(ssml_text=ssml, speaker=speaker, sample_rate=sample_rate)
    except TypeError:
        wav = model.apply_tts(text=str(text), speaker=speaker, sample_rate=sample_rate)
    except AssertionError as e:
        # Если выбранный voice не поддержан конкретной моделью, пробуем русские fallback-голоса.
        fallback_speakers = ['baya', 'kseniya', 'xenia', 'eugene']
        last_err = e
        for sp in fallback_speakers:
            try:
                print(f'voice fallback: {speaker} -> {sp}')
                wav = model.apply_tts(ssml_text=ssml, speaker=sp, sample_rate=sample_rate)
                break
            except Exception as err:
                last_err = err
        else:
            raise last_err

    wav = _as_2d_float_tensor(wav)
    wav = normalize_peak(wav * SPEECH_GAIN)
    return wav


def wav_duration_sec(wav):
    return wav.shape[1] / sample_rate

clip_rows = []
for _, row in comments_df.iterrows():
    period = int(row['period'])
    period_dir = CLIPS_DIR / f'period_{period}'
    period_dir.mkdir(parents=True, exist_ok=True)
    out_name = f"period{period}_{sec_to_stamp(row['t_start_sec'])}_{int(row['comment_idx']):04d}.wav"
    out_path = period_dir / out_name
    if out_path.exists() and not REGENERATE_CLIPS:
        wav, sr = torchaudio.load(str(out_path))
        if sr != sample_rate:
            wav = torchaudio.functional.resample(wav, sr, sample_rate)
    else:
        wav = synthesize_text(row['commentary'])
        torchaudio.save(str(out_path), wav.detach().cpu(), sample_rate)
    clip_rows.append({
        'comment_idx': int(row['comment_idx']),
        'period': period,
        't_start': row['t_start'],
        't_start_sec': float(row['t_start_sec']),
        'tts_duration_sec': round(wav_duration_sec(wav), 3),
        'clip_path': str(out_path),
        'commentary': row['commentary'],
    })

clips_df = pd.DataFrame(clip_rows)
clips_df.to_csv(TABLES_DIR / 'tts_clips_index.csv', index=False)
print('clips:', len(clips_df))
display(clips_df.head())




## 5. Быстрая проверка первых реплик


In [ ]:
for p in clips_df['clip_path'].head(3):
    print(Path(p).name)
    display(Audio(p))


## 6. Склейка с шумом в паузах


In [34]:
def make_soft_noise(duration_sec, sample_rate, rms=0.002, seed=42):
    n = max(1, int(math.ceil(duration_sec * sample_rate)))
    gen = torch.Generator(device=DEVICE).manual_seed(seed) if DEVICE.type == 'cuda' else torch.Generator().manual_seed(seed)
    noise = torch.randn(1, 1, n, generator=gen, device=DEVICE)
    win = max(3, int(sample_rate * 0.015))
    if win % 2 == 0:
        win += 1
    kernel = torch.ones(1, 1, win, device=DEVICE) / win
    smooth = F.conv1d(noise, kernel, padding=win // 2).view(1, -1)
    std = smooth.std().item()
    if std > 0:
        smooth = smooth / std * rms
    return smooth.to(torch.float32)


def load_clip(path):
    wav, sr = torchaudio.load(str(path))
    if sr != sample_rate:
        wav = torchaudio.functional.resample(wav, sr, sample_rate)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    return wav.to(torch.float32).to(DEVICE)


def overlay_clip(track, clip, start_sec):
    start = int(round(start_sec * sample_rate))
    end = start + clip.shape[1]
    if end > track.shape[1]:
        track = F.pad(track, (0, end - track.shape[1]))
    track[:, start:end] += clip
    return track, end / sample_rate

placement_rows = []
period_track_paths = []

for period in sorted(clips_df['period'].unique()):
    sub = clips_df[clips_df['period'] == period].sort_values('t_start_sec').reset_index(drop=True)
    base_duration = max(period_durations.get(int(period), 0.0), float((sub['t_start_sec'] + sub['tts_duration_sec']).max()) + TAIL_SEC)
    track = make_soft_noise(base_duration, sample_rate, rms=NOISE_RMS, seed=RANDOM_SEED + int(period))
    last_end_sec = 0.0
    for _, r in sub.iterrows():
        requested_start = float(r['t_start_sec'])
        place_start = max(requested_start, last_end_sec + MIN_GAP_SEC) if AVOID_OVERLAP else requested_start
        clip = load_clip(r['clip_path'])
        track, actual_end = overlay_clip(track, clip, place_start)
        last_end_sec = actual_end
        placement_rows.append({
            'comment_idx': int(r['comment_idx']),
            'period': int(period),
            'requested_start_sec': round(requested_start, 3),
            'placed_start_sec': round(place_start, 3),
            'shift_sec': round(place_start - requested_start, 3),
            'tts_duration_sec': round(clip.shape[1] / sample_rate, 3),
            'placed_end_sec': round(actual_end, 3),
            'clip_path': r['clip_path'],
            'commentary': r['commentary'],
        })
    track = torch.clamp(track, -0.99, 0.99)
    out_path = TRACKS_DIR / f'match_{MATCH_ID}_{STRATEGY}_period{int(period)}_with_noise.wav'
    torchaudio.save(str(out_path), track.detach().cpu(), sample_rate)
    period_track_paths.append(out_path)
    print('saved:', out_path, f'{track.shape[1] / sample_rate:.1f} sec')

placement_df = pd.DataFrame(placement_rows)
placement_df.to_csv(TABLES_DIR / 'placement_index.csv', index=False)
display(placement_df.head())
print('max shift sec:', placement_df['shift_sec'].max() if len(placement_df) else 0)


saved: /kaggle/working/outputs/audio_match_3930158_related_stop_cap_silero_xfast/stitched_tracks/match_3930158_related_stop_cap_period1_with_noise.wav 2927.4 sec
saved: /kaggle/working/outputs/audio_match_3930158_related_stop_cap_silero_xfast/stitched_tracks/match_3930158_related_stop_cap_period2_with_noise.wav 2922.8 sec


,comment_idx,period,requested_start_sec,placed_start_sec,shift_sec,tts_duration_sec,placed_end_sec,clip_path,commentary
0,1,1,1.191,1.191,0.000,8.600,9.791,/kaggle/working/outputs/audio_match_3930158_re...,"Хавертц начинает игру с центра поля, отдавая п..."
1,2,1,7.397,9.941,2.544,4.862,14.803,/kaggle/working/outputs/audio_match_3930158_re...,Хавертц делает пас низом после начального удар...
2,3,1,8.713,14.954,6.241,3.062,18.016,/kaggle/working/outputs/audio_match_3930158_re...,"Хендери делает передачу, но её блокирует Мюсиала."
3,4,1,10.048,18.166,8.118,2.675,20.841,/kaggle/working/outputs/audio_match_3930158_re...,Гендри блокирует удар после начального удара.
4,5,1,10.987,20.991,10.004,5.300,26.291,/kaggle/working/outputs/audio_match_3930158_re...,МакГрегор с первых секунд демонстрирует мастер...


max shift sec: 75.777


In [35]:
waves = []
for i, p in enumerate(period_track_paths):
    waves.append(load_clip(p))
    if i < len(period_track_paths) - 1 and HALF_GAP_SEC > 0:
        waves.append(make_soft_noise(HALF_GAP_SEC, sample_rate, rms=NOISE_RMS, seed=RANDOM_SEED + 100 + i))

if waves:
    full = torch.clamp(torch.cat(waves, dim=1), -0.99, 0.99)
    full_path = TRACKS_DIR / f'match_{MATCH_ID}_{STRATEGY}_full_with_noise.wav'
    torchaudio.save(str(full_path), full.detach().cpu(), sample_rate)
    print('saved full:', full_path, f'{full.shape[1] / sample_rate:.1f} sec')
else:
    full_path = None


saved full: /kaggle/working/outputs/audio_match_3930158_related_stop_cap_silero_xfast/stitched_tracks/match_3930158_related_stop_cap_full_with_noise.wav 5856.2 sec


## 7. Скачать результат

В Kaggle справа открой `/kaggle/working/outputs/...` или скачай zip из следующей ячейки.


In [36]:
import shutil
zip_base = BASE_DIR / f'audio_match_{MATCH_ID}_{STRATEGY}_silero_xfast'
zip_path = shutil.make_archive(str(zip_base), 'zip', AUDIO_OUT_DIR)
print('zip:', zip_path)


KeyboardInterrupt: 

In [ ]:
for p in period_track_paths:
    print(p.name)
    display(Audio(str(p)))

if full_path:
    print(full_path.name)
    display(Audio(str(full_path)))


match_3930158_related_stop_cap_period1_with_noise.wav
